In [18]:
# @title
# install the libraries we need
!pip install firebase pdfplumber nltk requests


# connect to firebase
from firebase import firebase

DATABASE_URL = "https://agrosense-cc792-default-rtdb.firebaseio.com/"

FBconn = firebase.FirebaseApplication(DATABASE_URL, None)
print("Firebase connected.")


import io
import requests

COLLECTION_NAME = "rosemary"

# maps each pdf filename to its Google Drive share link
PDF_DRIVE_LINKS = {
    "pone.0273367.pdf":
        "https://drive.google.com/file/d/1fg_9wOyZ6lHVX5VG337Pd6DJMZGH0bRU/view?usp=sharing",
    "Plant Pathology - 2010 - Park - First Korean report of rosemary powdery mildew caused by Golovinomyces biocellatus.pdf":
        "https://drive.google.com/file/d/11HNSROC9hOueLf3b07b9LcFeRLbC7dxU/view?usp=sharing",
    "1-s2.0-S0926669019309033-main.pdf":
        "https://drive.google.com/file/d/1x3zf3BaXk9LQGH-OaI2kMenv3uTrwT10/view?usp=sharing",
    "1-s2.0-S0926669024012925-main.pdf":
        "https://drive.google.com/file/d/1R38CxXribjt8b2q-PP4R6dvkZIuyroRB/view?usp=sharing",
    "hortsci-article-p208.pdf":
        "https://drive.google.com/file/d/1_yM_Q7BWr0wN1WLhfkGaO2GJcQBKPiw4/view?usp=sharing",
}

# the human-readable article title for each pdf. this is what we store in the
# `fname` field so the Search tab can show a real title instead of a filename.
PDF_TITLES = {
    "pone.0273367.pdf":
        "Seasonal Changes and Distribution Variations in Rosemary Oils",
    "Plant Pathology - 2010 - Park - First Korean report of rosemary powdery mildew caused by Golovinomyces biocellatus.pdf":
        "First Korean Report of Rosemary Powdery Mildew caused by Golovinomyces biocellatus",
    "1-s2.0-S0926669019309033-main.pdf":
        "Improving Water Use Efficiency through Drought Stress and Salicylic Acid in Rosmarinus officinalis",
    "1-s2.0-S0926669024012925-main.pdf":
        "Secondary Metabolites and Biomass in Rosemary",
    "hortsci-article-p208.pdf":
        "Drought Tolerance and Leaf Physiology in Rosemary",
}

# the 20 terms we want to index
SPECIAL_WORDS = [
    "Essential oils",
    "Drought stress",
    "Salicylic acid",
    "Secondary metabolites",
    "Rosemary",
    "Antibacterial activity",
    "Morphology",
    "Physiology",
    "Biomass",
    "Terpenes",
    "Leaf temperature",
    "Water use efficiency",
    "Stomatal conductance",
    "Chemotaxonomy",
    "Seasonal changes",
    "Trichomes",
    "Powdery mildew",
    "Genotype",
    "Tolerance",
    "Sustainability",
]

# stop words we decided to filter out before indexing.
# we chose these because they show up in every sentence and don't
# tell us anything useful about the topic of the paper.
# for example "the", "and", "is" appear hundreds of times in any
# article and would just add noise to the index.
# we also removed common academic words like "figure", "table",
# "results" because they appear in every paper regardless of topic.
STOP_WORDS = {
    # basic english words that carry no meaning on their own
    "a", "an", "the",
    "of", "in", "on", "at", "to", "for", "from", "by", "with",
    "into", "through", "during", "before", "after", "above",
    "below", "between", "out", "off", "over", "under", "again",
    "and", "or", "but", "so", "yet", "nor",
    "it", "its", "this", "that", "these", "those",
    "they", "their", "them", "we", "our", "i", "my",
    "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will",
    "would", "could", "should", "may", "might", "shall",
    # words that appear in every scientific paper but say nothing about content
    "et", "al", "fig", "figure", "table", "results",
    "conclusion", "abstract", "study", "studies",
    "also", "however", "therefore", "thus", "whereas",
    "using", "used", "use", "based", "shown", "found",
}

print("stop words we are using:", sorted(STOP_WORDS))
print(f"total: {len(STOP_WORDS)} words")


# build the index
# we use lemmatization instead of stemming because it gives cleaner results.
# for example "oils" becomes "oil" and "stresses" becomes "stress"
# but the word still looks like a real word, unlike stemming which
# can cut words too aggressively (e.g. "leaves" -> "leav")
import pdfplumber
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("wordnet",   quiet=True)
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4",   quiet=True)

lemmatizer = WordNetLemmatizer()

def lemmatize_phrase(phrase):
    tokens = word_tokenize(phrase.lower())
    return " ".join(lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS)

def download_pdf(share_link):
    """Download a Google Drive shared PDF into a BytesIO buffer."""
    file_id = share_link.split("/d/")[1].split("/")[0]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    session = requests.Session()
    response = session.get(url, stream=True)
    # large files get a virus-scan confirmation page — handle that token
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            response = session.get(url, params={"confirm": value}, stream=True)
            break
    response.raise_for_status()
    return io.BytesIO(response.content)

def extract_and_clean(pdf_buffer):
    raw_pages, paragraphs = [], []
    with pdfplumber.open(pdf_buffer) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                raw_pages.append(text)
                paragraphs.extend([p.strip() for p in text.split("\n\n") if p.strip()])

    full_text  = " ".join(raw_pages).lower()
    tokens     = word_tokenize(full_text)
    clean_text = " ".join(
        lemmatizer.lemmatize(t) for t in tokens
        if t.isalpha() and t not in STOP_WORDS
    )
    return clean_text, paragraphs

print(f"found {len(PDF_DRIVE_LINKS)} pdf files: {list(PDF_DRIVE_LINKS.keys())}")

pdf_texts      = {}
pdf_paragraphs = {}

for fname, share_link in PDF_DRIVE_LINKS.items():
    print(f"downloading: {fname}")
    pdf_buffer = download_pdf(share_link)
    pdf_texts[fname], pdf_paragraphs[fname] = extract_and_clean(pdf_buffer)

# for each term check which pdfs contain it and save the list
index = {}
for word in SPECIAL_WORDS:
    lemmatized_word = lemmatize_phrase(word)
    matching_docs   = [
        fname for fname, text in pdf_texts.items()
        if lemmatized_word in text
    ]
    index[word] = matching_docs

print("\nindex result:")
for term, docs in index.items():
    print(f"  {term}: {docs if docs else '(not found in any pdf)'}")

# upload each term to firebase — DocIDs use drive links, Snippets are parallel list
for term, doc_names in index.items():
    drive_links = []
    titles      = []
    snippets    = []
    lem_term    = lemmatize_phrase(term)
    for fname in doc_names:
        if fname not in PDF_DRIVE_LINKS:
            continue
        link = PDF_DRIVE_LINKS[fname]
        drive_links.append(link)
        # store the article title (fall back to the filename if we don't have one),
        # parallel to DocIDs and in the same order as the links
        titles.append(PDF_TITLES.get(fname, fname))
        paras  = pdf_paragraphs.get(fname, [])
        scored = sorted(
            [(sum(1 for w in lem_term.split() if w in p.lower()), p)
             for p in paras if len(p) > 30],
            reverse=True
        )
        top = [p for s, p in scored if s > 0][:2]
        snippets.append((" … ".join(top))[:400] if top else "")
    data_to_upload = {"term": term, "DocIDs": drive_links, "fname": titles, "Snippets": snippets}
    result = FBconn.post(f"/{COLLECTION_NAME}/", data_to_upload)
print(f"\nfinished. uploaded {len(index)} terms to firebase with snippets.")

Firebase connected.
stop words we are using: ['a', 'above', 'abstract', 'after', 'again', 'al', 'also', 'an', 'and', 'are', 'at', 'based', 'be', 'been', 'before', 'being', 'below', 'between', 'but', 'by', 'conclusion', 'could', 'did', 'do', 'does', 'during', 'et', 'fig', 'figure', 'for', 'found', 'from', 'had', 'has', 'have', 'however', 'i', 'in', 'into', 'is', 'it', 'its', 'may', 'might', 'my', 'nor', 'of', 'off', 'on', 'or', 'our', 'out', 'over', 'results', 'shall', 'should', 'shown', 'so', 'studies', 'study', 'table', 'that', 'the', 'their', 'them', 'therefore', 'these', 'they', 'this', 'those', 'through', 'thus', 'to', 'under', 'use', 'used', 'using', 'was', 'we', 'were', 'whereas', 'will', 'with', 'would', 'yet']
total: 85 words
found 5 pdf files: ['pone.0273367.pdf', 'Plant Pathology - 2010 - Park - First Korean report of rosemary powdery mildew caused by Golovinomyces biocellatus.pdf', '1-s2.0-S0926669019309033-main.pdf', '1-s2.0-S0926669024012925-main.pdf', 'hortsci-article-p20

In [19]:
# @title
# connect to firebase
!pip install firebase pdfplumber requests
from firebase import firebase

DATABASE_URL = "https://agrosense-cc792-default-rtdb.firebaseio.com/"

FBconn = firebase.FirebaseApplication(DATABASE_URL, None)
print("Firebase connected.")


import io
import requests
import pdfplumber


def search_firebase(query, collection="rosemary"):
    """Search Firebase index for entries whose term matches any query word.
    Returns a list of unique drive links, or None if nothing matched."""
    query_words = query.lower().split()
    all_entries = FBconn.get(f"/{collection}/", None)
    if not all_entries:
        return None

    matched_links = set()
    for entry in all_entries.values():
        term = entry.get("term", "").lower()
        if any(w in term for w in query_words):
            for link in entry.get("DocIDs", []):
                matched_links.add(link)

    return list(matched_links) if matched_links else None


def _download_pdf(share_link):
    file_id = share_link.split("/d/")[1].split("/")[0]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    session = requests.Session()
    response = session.get(url, stream=True)
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            response = session.get(url, params={"confirm": value}, stream=True)
            break
    response.raise_for_status()
    return io.BytesIO(response.content)


def _extract_paragraphs(pdf_buffer):
    paragraphs = []
    with pdfplumber.open(pdf_buffer) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                paragraphs.extend([p.strip() for p in text.split("\n\n") if p.strip()])
    return paragraphs


def rag_query(query, top_k_paragraphs=2):
    """Search Firebase for matching docs, download them, and return top paragraphs.
    Returns a list of {"link": ..., "paragraphs": [...]} dicts, or None if no match."""
    links = search_firebase(query)
    if links is None:
        return f"answer this question: {query} by your knowledge"

    query_lower = query.lower()
    results = []

    for link in links:
        pdf_buffer  = _download_pdf(link)
        paragraphs  = _extract_paragraphs(pdf_buffer)

        scored = [
            (sum(1 for w in query_lower.split() if w in para.lower()), para)
            for para in paragraphs
        ]
        scored = [(s, p) for s, p in scored if s > 0]
        scored.sort(reverse=True)

        if scored:
            results.append({
                "link": link,
                "paragraphs": [para for _, para in scored[:top_k_paragraphs]],
            })

    return f"answer this question: {query} with this result {results}" if results else f"answer this question: {query} by your knowledge"

Firebase connected.


In [20]:
#@title
# install the libraries we need
!pip install firebase pdfplumber requests
# connect to firebase
from firebase import firebase

DATABASE_URL = "https://agrosense-cc792-default-rtdb.firebaseio.com/"

FBconn = firebase.FirebaseApplication(DATABASE_URL, None)
print("Firebase connected.")


import io
import requests
import pdfplumber


def search_firebase(query, collection="rosemary"):
    """Search Firebase index for entries whose term matches any query word.
    Returns a list of unique drive links, or None if nothing matched."""
    query_words = query.lower().split()
    all_entries = FBconn.get(f"/{collection}/", None)
    if not all_entries:
        return None

    matched_links = set()
    for entry in all_entries.values():
        term = entry.get("term", "").lower()
        if any(w in term for w in query_words):
            for link in entry.get("DocIDs", []):
                matched_links.add(link)

    return list(matched_links) if matched_links else None


def _download_pdf(share_link):
    file_id = share_link.split("/d/")[1].split("/")[0]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    session = requests.Session()
    response = session.get(url, stream=True)
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            response = session.get(url, params={"confirm": value}, stream=True)
            break
    response.raise_for_status()
    return io.BytesIO(response.content)


def _extract_paragraphs(pdf_buffer):
    paragraphs = []
    with pdfplumber.open(pdf_buffer) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                paragraphs.extend([p.strip() for p in text.split("\n\n") if p.strip()])
    return paragraphs


def rag_query(query, top_k_paragraphs=2):
    """Search Firebase for matching docs, download them, and return top paragraphs.
    Returns a list of {"link": ..., "paragraphs": [...]} dicts, or None if no match."""
    links = search_firebase(query)
    if links is None:
        return f"answer this question: {query} by your knowledge"

    query_lower = query.lower()
    results = []

    for link in links:
        pdf_buffer  = _download_pdf(link)
        paragraphs  = _extract_paragraphs(pdf_buffer)

        scored = [
            (sum(1 for w in query_lower.split() if w in para.lower()), para)
            for para in paragraphs
        ]
        scored = [(s, p) for s, p in scored if s > 0]
        scored.sort(reverse=True)

        if scored:
            results.append({
                "link": link,
                "paragraphs": [para for _, para in scored[:top_k_paragraphs]],
            })

    return f"answer this question: {query} with this result {results}" if results else f"answer this question: {query} by your knowledge"

# @title IoT Sensors - read data from the course server
# This is the backend for the Sensors tab.
# We don't use Adafruit keys anymore. Instead we ask the course server and it
# gives us the data (the server keeps the API key safe so we don't leak it).
# We only choose which feed to read (humidity / soil / temperature / json)
# and how many samples we want.
#
# the request looks like this:
#   GET {BASE_URL}/history?feed=humidity&limit=10
# and we get back something like: {"data": [{"value": ..., "created_at": ...}, ...]}
#
# two things to remember:
# - the server only lets us READ, so "Save reading"/"Simulate" just keeps the
#   reading on our side (it doesn't really get sent to the server).
# - the server goes to sleep when nobody uses it, so the first request can take
#   a long time (like 30-60 sec) until it wakes up. that's normal.
import json
from collections import deque
from datetime import datetime
from google.colab import output
from IPython.display import JSON

# ── server settings ────────────────────────────────────
BASE_URL = "https://server-cloud-v645.onrender.com"   # the course server
HISTORY_LIMIT = 12                                     # how many samples to show
REQ_TIMEOUT = 90                                       # wait up to 90 sec (server might be asleep)

# readings the user adds with the form. we keep them only here because the
# server is read-only and won't save them for us.
_local_readings = deque(maxlen=50)


def _num(value, default="N/A"):
    # make the value a number with 1 digit after the dot.
    # if it's empty or not a number, just return "N/A".
    try:
        return round(float(value), 1)
    except (TypeError, ValueError):
        return default


def _fmt_time(created_at):
    # the server sends a long date like 2024-01-15T12:34:56Z,
    # this makes it short like "01-15 12:34" so it fits in the table.
    if not created_at:
        return ""
    try:
        return datetime.strptime(str(created_at)[:19], "%Y-%m-%dT%H:%M:%S").strftime("%m-%d %H:%M")
    except (ValueError, TypeError):
        return str(created_at)


def _history(feed, limit):
    # ask the server for one feed and return its samples (newest one first)
    resp = requests.get(f"{BASE_URL}/history", params={"feed": feed, "limit": limit},
                        timeout=REQ_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    return data.get("data", []) if isinstance(data, dict) else []


def _rows_from_json_feed(limit):
    # first try the "json" feed. if each sample is a json object with all the
    # values inside, we can build the whole table with just one request (faster).
    # if it's not like that, we return None and use the separate feeds instead.
    rows = []
    for s in _history("json", limit):              # newest first
        raw = s.get("value", "")
        try:
            d = json.loads(raw) if isinstance(raw, str) else raw
        except (json.JSONDecodeError, TypeError):
            d = None
        if not isinstance(d, dict):
            return None
        rows.append({
            "timestamp":   _fmt_time(s.get("created_at")),
            "temperature": _num(d.get("temperature")),
            "humidity":    _num(d.get("humidity")),
            "soil":        _num(d.get("soil")),
            "light":       _num(d.get("light")),
        })
    rows.reverse()                                 # flip to oldest first (the table wants last = newest)
    return rows


def _rows_from_separate_feeds(limit):
    # get temperature, humidity and soil one feed at a time and put them together.
    # there is no "light" feed on the server, so we just put "N/A" for light.
    temps = _history("temperature", limit)         # each list is newest first
    hums  = _history("humidity", limit)
    soils = _history("soil", limit)
    n = max(len(temps), len(hums), len(soils))
    rows = []
    for i in range(n):                             # i=0 is the newest of each feed, so they line up
        t = temps[i] if i < len(temps) else {}
        h = hums[i]  if i < len(hums)  else {}
        s = soils[i] if i < len(soils) else {}
        ts = t.get("created_at") or h.get("created_at") or s.get("created_at")
        rows.append({
            "timestamp":   _fmt_time(ts),
            "temperature": _num(t.get("value")),
            "humidity":    _num(h.get("value")),
            "soil":        _num(s.get("value")),
            "light":       "N/A",
        })
    rows.reverse()                                 # oldest first
    return rows


# these two functions get called from the web page (the Sensors tab).
# the arguments come in as one json string.
def get_sensor_data(args_json="{}"):
    # runs when we open the tab or press Refresh
    try:
        rows = _rows_from_json_feed(HISTORY_LIMIT)
        if not rows:                               # json feed was empty/not objects -> use separate feeds
            rows = _rows_from_separate_feeds(HISTORY_LIMIT)
    except Exception as e:
        print("get_sensor_data error:", e)
        rows = []
    rows = (rows or []) + list(_local_readings)     # add the readings we saved locally
    return JSON(rows)


def add_sensor_reading(args_json="{}"):
    # runs when we press "Save reading" / "Simulate".
    # the server is read-only, so we only keep the reading here on our side.
    try:
        payload = json.loads(args_json) if isinstance(args_json, str) else (args_json or {})
        _local_readings.append({
            "timestamp":   datetime.now().strftime("%m-%d %H:%M"),
            "humidity":    _num(payload.get("humidity")),
            "temperature": _num(payload.get("temperature")),
            "light":       _num(payload.get("light")),
            "soil":        _num(payload.get("soil")),
        })
        return JSON({"success": True, "local": True})
    except Exception as e:
        print("add_sensor_reading error:", e)
        return JSON({"success": False, "error": str(e)})


# tell Colab that the web page is allowed to call these two functions
output.register_callback("get_sensor_data", get_sensor_data)
output.register_callback("add_sensor_reading", add_sensor_reading)
print(f"IoT sensor callbacks registered (server: {BASE_URL}).")

# @title Daily Care Tasks - turn the latest sensor reading into a to-do list
# This is the "Daily Care Tasks" feature. The idea: look at the newest sensor
# reading, compare it to what rosemary actually likes, and give the manager a
# short list of concrete things to do today (water it, move it to the light...).
# It also keeps a small "streak" so the manager can see how many days in a row
# they kept up with the care tasks.
from datetime import date, timedelta

# Best growing conditions for rosemary (Rosmarinus officinalis), which is a
# Mediterranean evergreen herb: lots of sun, dryish well-drained soil, mild
# temperatures and air that isn't too humid. The numbers below are practical
# ranges from common horticultural guides (RHS / university extension grow
# guides) and are written as (min, max).
OPTIMAL_RANGES = {
    "humidity":    (40, 60),       # % - moderate; humid air (>~65%) invites powdery mildew / rot
    "temperature": (15, 27),       # °C - comfortable Mediterranean range; grows best here
    "light":       (2000, 100000), # lux - rosemary loves full sun (~10k-100k lux outdoors). Indoor
                                   #       sensors read much lower, so the low bound flags a too-dark
                                   #       spot. May need calibrating to your specific sensor.
    "soil":        (30, 50),       # % - keep on the dry side; let it dry out between waterings
}

# the exact wording we show for each problem. key = (which sensor, low or high).
TASK_TEMPLATES = {
    ("soil",        "low"):  "Water the plant — soil moisture is low ({v}%)",
    ("soil",        "high"): "Hold off watering — soil is too wet ({v}%)",
    ("light",       "low"):  "Move to a brighter spot — only {v} lux today",
    ("light",       "high"): "Give some shade — light is very strong ({v} lux)",
    ("humidity",    "high"): "Check for mildew — humidity is high ({v}%)",
    ("humidity",    "low"):  "Mist lightly / group plants — humidity is low ({v}%)",
    ("temperature", "low"):  "Move somewhere warmer — temperature is low ({v}°C)",
    ("temperature", "high"): "Cool down / ventilate — temperature is high ({v}°C)",
}


def _to_float(value):
    # try to read a value as a number. sensor values can be "N/A", so return
    # None for anything that isn't really a number and we just skip it.
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _check_ranges(latest_reading):
    # compare each sensor value to its optimal range.
    # returns a list of (sensor, value, "low"/"high") for everything out of range.
    issues = []
    for metric, (low, high) in OPTIMAL_RANGES.items():
        v = _to_float(latest_reading.get(metric))
        if v is None:
            continue                       # no real reading for this sensor, skip it
        if v < low:
            issues.append((metric, v, "low"))
        elif v > high:
            issues.append((metric, v, "high"))
    return issues


def generate_daily_tasks(latest_reading):
    # main feature function: take the newest reading (dict with humidity,
    # temperature, light, soil) and build the list of task strings for today.
    # if nothing has a real number, we can't say anything useful.
    if all(_to_float(latest_reading.get(m)) is None for m in OPTIMAL_RANGES):
        return ["No sensor data yet — check the Sensors tab"]

    tasks = []
    for metric, value, status in _check_ranges(latest_reading):
        template = TASK_TEMPLATES.get((metric, status))
        if template:
            # light is shown as a whole number (lux), the rest with 1 decimal
            shown = int(value) if metric == "light" else round(value, 1)
            tasks.append(template.format(v=shown))

    # everything inside the optimal ranges -> one happy message, no chores
    if not tasks:
        return ["All good — no action needed today 🌿"]
    return tasks


def generate_alerts(latest_reading):
    # short warning lines for the dashboard's "Alerts" card (just the facts,
    # while generate_daily_tasks gives the actions to take).
    labels = {"humidity": "Humidity", "temperature": "Temperature",
              "light": "Light", "soil": "Soil moisture"}
    units  = {"humidity": "%", "temperature": "°C", "light": " lux", "soil": "%"}
    alerts = []
    for metric, value, status in _check_ranges(latest_reading):
        shown = int(value) if metric == "light" else round(value, 1)
        alerts.append(f"{labels[metric]} {status}: {shown}{units[metric]}")
    return alerts


def compute_health(latest_reading):
    # graded 0-100 health score. 50 is the neutral "standard": a sensor sitting
    # right on the edge of its optimal range scores 50. The closer a reading is
    # to the middle of its ideal range, the higher it climbs toward 100; the
    # further it strays outside the range, the lower it drops toward 0.
    # The overall score is the average across every sensor we have a reading for.
    scores = []
    for metric, (low, high) in OPTIMAL_RANGES.items():
        v = _to_float(latest_reading.get(metric))
        if v is None:
            continue
        width = high - low
        if width <= 0:
            # degenerate range: only the exact value is "ideal"
            scores.append(100.0 if v == low else 0.0)
            continue
        if low <= v <= high:
            # in range: 50 at the edges, 100 dead-centre.
            center = (low + high) / 2
            centered = 1 - abs(v - center) / (width / 2)   # 1 at centre, 0 at edge
            scores.append(50 + 50 * centered)
        else:
            # out of range: 50 just outside, 0 once a full range-width past the edge.
            dist = (low - v) if v < low else (v - high)
            over = min(dist / width, 1.0)                  # 0 at edge, 1 a width away
            scores.append(50 * (1 - over))
    return round(sum(scores) / len(scores)) if scores else 50


# ── streak counter ───────────────────────────────────
# how many days in a row the manager finished their care tasks.
# We keep it in this dict for now so it's simple. In a real product this would
# be saved to storage (e.g. Firebase) per plant/user, because right now it
# resets whenever the Colab kernel restarts.
_streak_state = {"count": 0, "last_done": None}   # last_done = "YYYY-MM-DD" or None


def mark_tasks_done():
    # call this when the manager marks today's tasks as done.
    # if they also did it yesterday -> streak grows. if they skipped a day (or
    # it's the first time) -> streak starts again at 1. doing it twice in the
    # same day doesn't count twice.
    today = date.today()
    last = _streak_state["last_done"]
    if last == today.isoformat():
        pass                                          # already counted today
    elif last == (today - timedelta(days=1)).isoformat():
        _streak_state["count"] += 1                   # consecutive day
    else:
        _streak_state["count"] = 1                    # first time or streak broken
    _streak_state["last_done"] = today.isoformat()
    return _streak_state["count"]


def current_streak():
    # the streak only "counts" if the last completion was today or yesterday.
    # if a whole day was missed the streak is considered broken (shows 0).
    last = _streak_state["last_done"]
    if not last:
        return 0
    today = date.today()
    if last in (today.isoformat(), (today - timedelta(days=1)).isoformat()):
        return _streak_state["count"]
    return 0


def _current_readings():
    # get the readings the same way the Sensors tab does (server + anything
    # added locally). reuses the helpers from the sensors cell above.
    try:
        rows = _rows_from_json_feed(HISTORY_LIMIT) or _rows_from_separate_feeds(HISTORY_LIMIT)
    except Exception as e:
        print("dashboard readings error:", e)
        rows = []
    return (rows or []) + list(_local_readings)


# ── Colab callbacks for the Dashboard tab ────────────
def get_dashboard_data(args_json="{}"):
    # everything the Dashboard tab needs in one go: the readings for the charts,
    # the latest values, the health score, alerts, today's tasks and the streak.
    readings = _current_readings()
    latest = readings[-1] if readings else {}      # last row = newest reading
    return JSON({
        "readings":     readings,
        "latest":       latest,
        "health_score": compute_health(latest),
        "alerts":       generate_alerts(latest),
        "tasks":        generate_daily_tasks(latest),
        "streak":       current_streak(),
    })


def complete_daily_tasks(args_json="{}"):
    # runs when the manager presses "Mark today complete" on the tasks card.
    streak = mark_tasks_done()
    return JSON({"success": True, "streak": streak})


output.register_callback("get_dashboard_data", get_dashboard_data)
output.register_callback("complete_daily_tasks", complete_daily_tasks)
print("Daily Care Tasks + dashboard callbacks registered.")


# @title Search tab backend - register the search_index callback
# The Search tab in the web page calls invokeFunction("search_index", ...) and
# expects back {"rag_results": [{fname, link, snippet, terms}, ...]}. Nothing was
# registered under that name, so the page showed "Function not found: search_index".
# This cell builds that response straight from the Firebase index: each entry's
# snippets are stored parallel to its DocIDs, so we read the preview text directly
# and never download or parse the PDFs here (that's what used to make it slow).


def search_index(args_json="{}"):
    # parse the query that the page sent as a JSON string
    try:
        params = json.loads(args_json) if isinstance(args_json, str) else (args_json or {})
        query = (params.get("query") or "").strip()
    except (json.JSONDecodeError, TypeError):
        query = ""
    if not query:
        return JSON({"rag_results": []})

    query_words = query.lower().split()
    all_entries = FBconn.get("/rosemary/", None) or {}

    # collect, per document link, the matched terms, a preview snippet and the
    # article title. DocIDs, fname (title) and Snippets are stored parallel in the
    # index, so there's no PDF download or parsing here - we just read them out.
    terms_by_link   = {}
    snippet_by_link = {}
    snippet_score   = {}   # query-overlap score of the snippet currently chosen per link
    title_by_link   = {}
    for entry in all_entries.values():
        term = entry.get("term", "")
        # how many query words this entry's term contains (0 -> not a match)
        term_score = sum(1 for w in query_words if w in term.lower())
        if term_score == 0:
            continue
        links    = entry.get("DocIDs", []) or []
        # the index writes "Snippets"; tolerate the lowercase variants too
        snippets = entry.get("Snippets") or entry.get("snippets") or entry.get("snippts") or []
        titles   = entry.get("fname", []) or []
        for i, link in enumerate(links):
            terms_by_link.setdefault(link, set()).add(term)
            snippet = snippets[i] if i < len(snippets) else ""
            # keep the snippet whose term best overlaps the query (ties: first seen)
            if snippet and term_score > snippet_score.get(link, 0):
                snippet_by_link[link] = snippet
                snippet_score[link]   = term_score
            title = titles[i] if i < len(titles) else ""
            if title and not title_by_link.get(link):           # title is per-doc, first non-empty
                title_by_link[link] = title

    results = []
    for link, terms in terms_by_link.items():
        # use the stored article title; fall back to the Drive file id if missing
        fname = title_by_link.get(link)
        if not fname:
            try:
                fname = link.split("/d/")[1].split("/")[0]
            except (IndexError, AttributeError):
                fname = ""

        results.append({
            "link":    link,
            "fname":   fname,
            "snippet": snippet_by_link.get(link, ""),
            "terms":   sorted(terms),
        })

    return JSON({"rag_results": results})


output.register_callback("search_index", search_index)
print("Search callback registered.")

Firebase connected.
IoT sensor callbacks registered (server: https://server-cloud-v645.onrender.com).
Daily Care Tasks + dashboard callbacks registered.
Search callback registered.


In [21]:
# @title
from IPython.display import HTML

HTML('''
<!DOCTYPE html>
<html lang='en'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1'>
<script src='https://cdn.jsdelivr.net/npm/chart.js@4/dist/chart.umd.min.js'></script>
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Sans:ital,wght@0,300;0,400;0,500;0,600;1,400&family=DM+Mono:wght@400;500&display=swap');

  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

  :root {
    --green-dk: #2d6a4f;
    --green-md: #52b788;
    --green-lt: #d8f3dc;
    --green-bg: #f0f7f4;
    --green-xs: #b7e4c7;
    --text-main: #1a2e22;
    --text-muted: #4a6d57;
    --text-hint: #7a9e8a;
    --white: #ffffff;
    --border: rgba(45,106,79,0.18);
    --radius: 10px;
    --radius-sm: 6px;
  }

  body {
    font-family: 'DM Sans', sans-serif;
    background: var(--green-bg);
    color: var(--text-main);
    font-size: 14px;
    line-height: 1.6;
    min-height: 600px;
  }

  /* ── Shell ── */
  .app-shell { display: flex; flex-direction: column; min-height: 600px; }

  /* ── Header ── */
  .app-header {
    background: var(--green-dk);
    padding: 12px 20px 0;
    flex-shrink: 0;
  }
  .app-brand {
    display: flex; align-items: center; gap: 10px;
    color: var(--green-lt); font-size: 15px; font-weight: 600;
    margin-bottom: 12px;
  }
  .app-brand svg { flex-shrink: 0; }

  /* ── Tab bar ── */
  .tab-bar {
    display: flex; gap: 4px;
  }
  .tab-btn {
    flex: 1;
    background: transparent;
    border: none; border-radius: var(--radius-sm) var(--radius-sm) 0 0;
    color: rgba(216,243,220,0.65);
    font-family: 'DM Sans', sans-serif;
    font-size: 13px; font-weight: 500;
    padding: 9px 6px;
    cursor: pointer;
    transition: background 0.15s, color 0.15s;
    white-space: nowrap;
  }
  .tab-btn:hover  { background: rgba(255,255,255,0.08); color: var(--green-lt); }
  .tab-btn.active { background: var(--green-bg); color: var(--green-dk); font-weight: 600; }

  /* ── Content area ── */
  .tab-panels { flex: 1; padding: 20px; }
  .tab-panel   { display: none; }
  .tab-panel.active { display: block; }

  /* ── Cards ── */
  .card {
    background: var(--white);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 16px;
    margin-bottom: 14px;
  }
  .card-title {
    font-size: 13px; font-weight: 600;
    color: var(--green-dk); text-transform: uppercase;
    letter-spacing: 0.04em; margin-bottom: 12px;
  }

  /* ── Metric grid ── */
  .metric-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(110px,1fr)); gap: 10px; margin-bottom: 14px; }
  .metric-card {
    background: var(--green-lt);
    border-radius: var(--radius-sm);
    padding: 12px 10px;
    text-align: center;
  }
  .metric-label { font-size: 11px; color: var(--text-muted); font-weight: 500; margin-bottom: 4px; }
  .metric-value { font-size: 22px; font-weight: 600; color: var(--green-dk); font-variant-numeric: tabular-nums; }
  .metric-unit  { font-size: 11px; color: var(--text-hint); margin-left: 2px; }

  /* ── Buttons ── */
  .btn {
    display: inline-flex; align-items: center; gap: 6px;
    background: var(--green-dk); color: var(--white);
    border: none; border-radius: var(--radius-sm);
    font-family: 'DM Sans', sans-serif; font-size: 13px; font-weight: 500;
    padding: 8px 16px;
    cursor: pointer;
    transition: background 0.15s, transform 0.1s;
  }
  .btn:hover   { background: #245940; }
  .btn:active  { transform: scale(0.97); }
  .btn.outline {
    background: transparent; color: var(--green-dk);
    border: 1px solid var(--green-dk);
  }
  .btn.outline:hover { background: var(--green-lt); }
  .btn.sm { padding: 5px 10px; font-size: 12px; }

  /* ── Form elements ── */
  input[type='text'], input[type='number'], textarea, select {
    width: 100%;
    background: var(--green-bg);
    border: 1px solid var(--border);
    border-radius: var(--radius-sm);
    font-family: 'DM Sans', sans-serif;
    font-size: 13px; color: var(--text-main);
    padding: 8px 10px;
    transition: border-color 0.15s;
    outline: none;
  }
  input:focus, textarea:focus { border-color: var(--green-md); }
  label { font-size: 12px; color: var(--text-muted); font-weight: 500; margin-bottom: 4px; display: block; }
  .form-row { margin-bottom: 10px; }
  .form-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }

  /* ── Tables ── */
  .data-table { width: 100%; border-collapse: collapse; font-size: 12px; }
  .data-table th {
    background: var(--green-lt); color: var(--green-dk);
    font-weight: 600; font-size: 11px; text-transform: uppercase;
    letter-spacing: 0.04em; padding: 8px 10px; text-align: left;
  }
  .data-table td { padding: 7px 10px; border-bottom: 1px solid var(--border); color: var(--text-main); }
  .data-table tr:last-child td { border-bottom: none; }
  .data-table tr:hover td { background: var(--green-bg); }

  /* ── Status / alerts ── */
  .alert-item {
    background: #fffde7;
    border-left: 3px solid #f9a825;
    border-radius: 0 var(--radius-sm) var(--radius-sm) 0;
    padding: 8px 12px;
    font-size: 13px; color: #5a4000;
    margin-bottom: 6px;
  }
  .rag-box {
    background: #f0f9f3;
    border-left: 3px solid var(--green-md);
    border-radius: 0 var(--radius-sm) var(--radius-sm) 0;
    padding: 10px 14px;
    font-size: 12px; font-family: 'DM Mono', monospace;
    color: var(--text-muted);
    white-space: pre-wrap; word-break: break-word;
    max-height: 200px; overflow-y: auto;
    margin-bottom: 12px;
  }

  /* ── Daily care tasks ── */
  .task-list { list-style: none; }
  .task-item {
    display: flex; align-items: center; gap: 10px;
    padding: 8px 4px; border-bottom: 1px solid var(--border);
    font-size: 13px; color: var(--text-main);
  }
  .task-item:last-child { border-bottom: none; }
  .task-item input[type='checkbox'] {
    width: 16px; height: 16px; flex-shrink: 0;
    accent-color: var(--green-md); cursor: pointer;
  }
  .task-item.done span { text-decoration: line-through; color: var(--text-hint); }
  .task-all-good { padding: 8px 4px; color: var(--green-dk); font-weight: 500; }
  .streak-badge {
    display: inline-flex; align-items: center; gap: 5px;
    background: var(--green-lt); color: var(--green-dk);
    border-radius: 99px; padding: 4px 12px;
    font-size: 12px; font-weight: 600;
  }

  /* ── Health bar ── */
  .health-ring { text-align: center; margin-bottom: 16px; }
  .health-score-num { font-size: 52px; font-weight: 700; line-height: 1; }
  .health-bar-wrap { height: 10px; background: var(--green-lt); border-radius: 99px; margin: 8px auto; max-width: 300px; }
  .health-bar { height: 10px; border-radius: 99px; transition: width 0.6s ease; }
  .health-label { font-size: 12px; color: var(--text-hint); margin-top: 4px; }

  /* ── Drop zone ── */
  #drop-zone {
    border: 2px dashed var(--green-xs);
    border-radius: var(--radius);
    padding: 30px 20px;
    text-align: center;
    color: var(--text-hint);
    cursor: pointer;
    transition: border-color 0.2s, background 0.2s;
    margin-bottom: 12px;
  }
  #drop-zone.drag-over { border-color: var(--green-md); background: #e8f7ed; }
  #drop-zone svg { display: block; margin: 0 auto 8px; opacity: 0.5; }
  #preview-wrap { margin-bottom: 12px; }
  #preview-img  { max-width: 100%; max-height: 160px; border-radius: var(--radius-sm); display: none; }

  /* ── Gallery ── */
  #gallery { display: grid; grid-template-columns: repeat(auto-fill, minmax(130px,1fr)); gap: 8px; margin-top: 12px; }
  .gallery-item {
    background: var(--green-lt); border-radius: var(--radius-sm);
    padding: 8px; font-size: 11px; color: var(--text-muted);
    word-break: break-all;
  }
  .gallery-item b { display: block; color: var(--green-dk); margin-bottom: 2px; }

  /* ── Chip row ── */
  .chip-row { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 12px; }
  .chip {
    background: var(--green-lt); border: 1px solid var(--green-xs);
    color: var(--green-dk); border-radius: 99px;
    font-size: 12px; font-weight: 500;
    padding: 4px 12px; cursor: pointer;
    transition: background 0.12s;
  }
  .chip:hover { background: var(--green-xs); }

  /* ── Toast ── */
  #toast {
    position: fixed; bottom: 20px; right: 20px;
    background: var(--green-dk); color: var(--white);
    padding: 10px 18px; border-radius: var(--radius);
    font-size: 13px; font-weight: 500;
    opacity: 0; pointer-events: none;
    transition: opacity 0.3s;
    z-index: 9999;
  }
  #toast.show { opacity: 1; }

  /* ── Chart wrappers ── */
  .chart-wrap { position: relative; width: 100%; height: 200px; }
  .chart-legend { display: flex; gap: 16px; margin-bottom: 6px; flex-wrap: wrap; font-size: 12px; color: var(--text-muted); }
  .legend-dot { width: 10px; height: 10px; border-radius: 2px; display: inline-block; margin-right: 4px; }

  /* ── Spinner ── */
  @keyframes spin { to { transform: rotate(360deg); } }

  /* ── Misc ── */
  .sep { margin: 14px 0; border: none; border-top: 1px solid var(--border); }
  .inline-row { display: flex; gap: 8px; align-items: flex-end; }
  .inline-row input { flex: 1; }
  .success-msg { color: var(--green-dk); font-size: 13px; font-weight: 500; padding: 8px 0; }
  .section-sub { font-size: 12px; color: var(--text-hint); margin-bottom: 10px; }

  /* Search Result Specific Styles */
  .search-result-item {
    padding: 16px 0;
    border-bottom: 1px solid var(--border);
  }
  .search-result-item:last-child {
    border-bottom: none;
  }
  .search-result-breadcrumb {
    display: flex;
    align-items: center;
    gap: 6px;
    font-size: 12px;
    color: #5f6368;
    margin-bottom: 4px;
  }
  .search-result-pdf-icon {
    flex-shrink: 0;
    width: 14px;
    height: 14px;
    fill: #dc3545; /* Red color for PDF icon */
  }
  .search-result-link {
    font-size: 20px;
    font-weight: 400;
    color: #1a0dab;
    text-decoration: none;
    line-height: 1.3;
    display: block;
    margin-bottom: 5px;
  }
  .search-result-link:hover {
    text-decoration: underline;
  }
  .search-result-description {
    font-size: 14px;
    color: #4d5156;
    line-height: 1.6;
    margin: 0 0 8px;
  }
  .search-result-chip-container {
    display: flex;
    flex-wrap: wrap;
    gap: 4px;
  }
  .search-result-chip {
    background: var(--green-lt);
    border: 1px solid var(--green-xs);
    color: var(--green-dk);
    border-radius: 99px;
    font-size: 11px;
    font-weight: 500;
    padding: 2px 10px;
  }
</style>
</head>
<body>

<div id='toast'></div>

<div class='app-shell'>

  <!-- Header + tabs -->
  <header class='app-header'>
    <div class='app-brand'>
      <svg width='22' height='22' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.8'>
        <path d='M12 2C7 2 4 7 4 12c0 4.4 2.8 8.1 7 9.5V22h2v-.5C17.2 20.1 20 16.4 20 12c0-5-3-10-8-10z'/>
        <path d='M12 7v10M9 10l3-3 3 3'/>
      </svg>
      AgroSense — Rosemary Monitor
    </div>
    <nav class='tab-bar' role='tablist'>
      <button class='tab-btn active' role='tab' onclick='switchTab(0)' id='tab0'>📷 Upload</button>
      <button class='tab-btn'        role='tab' onclick='switchTab(1)' id='tab1'>📡 Sensors</button>
      <button class='tab-btn'        role='tab' onclick='switchTab(2)' id='tab2'>🔍 Search</button>
      <button class='tab-btn'        role='tab' onclick='switchTab(3)' id='tab3'>📊 Dashboard</button>
    </nav>
  </header>

  <main class='tab-panels'>

    <!-- ═══ TAB 0 — Upload ═══ -->
    <section class='tab-panel active' id='panel0' role='tabpanel'>
      <div class='card'>
        <p class='card-title'>Plant image upload</p>

        <div id='drop-zone' onclick='document.getElementById("fileInput").click()'
             ondragover='event.preventDefault();this.classList.add("drag-over")'
             ondragleave='this.classList.remove("drag-over")'
             ondrop='handleDrop(event)'>
          <svg width='36' height='36' viewBox='0 0 24 24' fill='none' stroke='#52b788' stroke-width='1.5'>
            <path d='M21 15v4a2 2 0 01-2 2H5a2 2 0 01-2-2v-4'/>
            <polyline points='17 8 12 3 7 8'/><line x1='12' y1='3' x2='12' y2='15'/>
          </svg>
          <span style='font-size:13px;'>Drag &amp; drop an image, or click to browse</span>
        </div>
        <input type='file' id='fileInput' accept='image/*' style='display:none' onchange='handleFile(this.files[0])'>

        <div id='preview-wrap'>
          <img id='preview-img' alt='Preview'>
        </div>

        <div class='form-row'>
          <label for='fname-input'>Filename</label>
          <input type='text' id='fname-input' placeholder='e.g. rosemary_north_bed_001.jpg'>
        </div>
        <div class='form-row'>
          <label for='notes-input'>Observations</label>
          <textarea id='notes-input' rows='3' placeholder='Describe leaf condition, growth stage, any anomalies…'></textarea>
        </div>

        <div style='display:flex;gap:8px;'>
          <button class='btn' onclick='doUpload()'>
            <svg width='14' height='14' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='2'>
              <polyline points='16 16 12 12 8 16'/><line x1='12' y1='12' x2='12' y2='21'/>
              <path d='M20.39 18.39A5 5 0 0018 9h-1.26A8 8 0 103 16.3'/>
            </svg>
            Upload image
          </button>
          <button class='btn outline' onclick='clearUpload()'>Clear</button>
        </div>

        <div id='upload-msg'></div>
      </div>

      <div class='card'>
        <p class='card-title'>Image gallery</p>
        <div id='gallery'><p style='font-size:12px;color:var(--text-hint);'>No images uploaded yet.</p></div>
      </div>
    </section>

    <!-- ═══ TAB 1 — Sensors ═══ -->
    <section class='tab-panel' id='panel1' role='tabpanel'>

      <div class='metric-grid' id='sensor-cards'>
        <div class='metric-card'><div class='metric-label'>Humidity</div><div class='metric-value' id='mc-hum'>—<span class='metric-unit'>%</span></div></div>
        <div class='metric-card'><div class='metric-label'>Temperature</div><div class='metric-value' id='mc-temp'>—<span class='metric-unit'>°C</span></div></div>
        <div class='metric-card'><div class='metric-label'>Light</div><div class='metric-value' id='mc-light'>—<span class='metric-unit'>lux</span></div></div>
        <div class='metric-card'><div class='metric-label'>Soil moisture</div><div class='metric-value' id='mc-soil'>—<span class='metric-unit'>%</span></div></div>
      </div>

      <div class='card'>
        <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:10px;'>
          <p class='card-title' style='margin:0;'>Sensor readings</p>
          <button class='btn sm' onclick='loadSensors()'>↻ Refresh</button>
        </div>
        <div style='overflow-x:auto;'>
          <table class='data-table' id='sensor-table'>
            <thead><tr><th>Time</th><th>Humidity %</th><th>Temp °C</th><th>Light lux</th><th>Soil %</th></tr></thead>
            <tbody id='sensor-tbody'><tr><td colspan='5' style='color:var(--text-hint);text-align:center;'>Loading…</td></tr></tbody>
          </table>
        </div>
      </div>

      <div class='card'>
        <p class='card-title'>Add / simulate reading</p>
        <div class='form-grid'>
          <div class='form-row'><label>Humidity (%)</label><input type='number' id='in-hum'  min='0' max='100' step='0.1' value='55'></div>
          <div class='form-row'><label>Temperature (°C)</label><input type='number' id='in-temp' min='0' max='50'  step='0.1' value='22'></div>
          <div class='form-row'><label>Light (lux)</label><input type='number' id='in-light' min='0' step='1'   value='400'></div>
          <div class='form-row'><label>Soil moisture (%)</label><input type='number' id='in-soil' min='0' max='100' step='0.1' value='42'></div>
        </div>
        <div style='display:flex;gap:8px;'>
          <button class='btn outline sm' onclick='simulateReading()'>🎲 Simulate</button>
          <button class='btn sm' onclick='saveReading()'>Save reading</button>
        </div>
        <div id='save-msg'></div>
      </div>
    </section>

    <!-- ═══ TAB 2 — Search ═══ -->
    <section class='tab-panel' id='panel2' role='tabpanel'>
      <div class='card'>
        <p class='card-title'>Search the index</p>
        <div class='inline-row form-row'>
          <input type='text' id='search-input' placeholder='e.g. essential oils, drought, terpenes…'
                 onkeydown='if(event.key==="Enter")doSearch()'>
          <button class='btn' onclick='doSearch()'>Search</button>
        </div>

        <div class='chip-row' id='chip-row'>
          <button class='chip' onclick='quickSearch("Rosemary")'>Rosemary</button>
          <button class='chip' onclick='quickSearch("Essential oils")'>Essential oils</button>
          <button class='chip' onclick='quickSearch("Terpenes")'>Terpenes</button>
          <button class='chip' onclick='quickSearch("Drought stress")'>Drought stress</button>
          <button class='chip' onclick='quickSearch("Powdery mildew")'>Powdery mildew</button>
          <button class='chip' onclick='quickSearch("Stomatal conductance")'>Stomatal conductance</button>
          <button class='chip' onclick='quickSearch("Salicylic acid")'>Salicylic acid</button>
        </div>

        <div id='rag-box' class='rag-box' style='display:none;'></div>

        <div id='search-results'>
          <p class='section-sub'>Enter a query above to search the document index.</p>
        </div>
      </div>
    </section>

    <!-- ═══ TAB 3 — Dashboard ═══ -->
    <section class='tab-panel' id='panel3' role='tabpanel'>

      <div style='display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:14px;'>
        <!-- Health score -->
        <div class='card'>
          <p class='card-title'>Plant health score</p>
          <div class='health-ring'>
            <div class='health-score-num' id='dash-health'>—</div>
            <div style='font-size:11px;color:var(--text-hint);'>/100</div>
            <div class='health-bar-wrap'><div class='health-bar' id='health-bar' style='width:0%'></div></div>
            <div class='health-label' id='health-label'>Loading…</div>
          </div>
        </div>

        <!-- Alerts -->
        <div class='card'>
          <p class='card-title'>Alerts</p>
          <div id='dash-alerts'><p style='font-size:12px;color:var(--text-hint);'>Loading…</p></div>
        </div>
      </div>

      <!-- Today's care tasks (sensor-driven checklist + streak) -->
      <div class='card'>
        <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:10px;'>
          <p class='card-title' style='margin:0;'>Today's tasks</p>
          <span class='streak-badge' id='streak-badge'>🔥 <span id='streak-count'>0</span>-day streak</span>
        </div>
        <ul class='task-list' id='task-list'>
          <li class='task-all-good'>Loading…</li>
        </ul>
        <div style='margin-top:10px;'>
          <button class='btn sm' id='complete-btn' onclick='markTasksDone()'>✓ Mark today complete</button>
        </div>
      </div>

      <!-- Latest metrics -->
      <div class='metric-grid' id='dash-cards'>
        <div class='metric-card'><div class='metric-label'>Humidity</div><div class='metric-value' id='d-hum'>—<span class='metric-unit'>%</span></div></div>
        <div class='metric-card'><div class='metric-label'>Temperature</div><div class='metric-value' id='d-temp'>—<span class='metric-unit'>°C</span></div></div>
        <div class='metric-card'><div class='metric-label'>Light</div><div class='metric-value' id='d-light'>—<span class='metric-unit'>lux</span></div></div>
        <div class='metric-card'><div class='metric-label'>Soil moisture</div><div class='metric-value' id='d-soil'>—<span class='metric-unit'>%</span></div></div>
      </div>

      <div class='card'>
        <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:8px;'>
          <p class='card-title' style='margin:0;'>Temperature &amp; humidity — last 12 readings</p>
        </div>
        <div class='chart-legend'>
          <span><span class='legend-dot' style='background:#2d6a4f;'></span>Temperature °C</span>
          <span><span class='legend-dot' style='background:#52b788;border:1.5px dashed #2d6a4f;'></span>Humidity %</span>
        </div>
        <div class='chart-wrap'><canvas id='chartTH' role='img' aria-label='Line chart of temperature and humidity over last 12 sensor readings'>Temperature and humidity trend data.</canvas></div>
      </div>

      <div class='card'>
        <p class='card-title' style='margin-bottom:8px;'>Light &amp; soil moisture — last 12 readings</p>
        <div class='chart-legend'>
          <span><span class='legend-dot' style='background:#b7e4c7;'></span>Light lux</span>
          <span><span class='legend-dot' style='background:#40916c;border:1.5px dashed #1b4332;'></span>Soil moisture %</span>
        </div>
        <div class='chart-wrap'><canvas id='chartLS' role='img' aria-label='Line chart of light and soil moisture over last 12 sensor readings'>Light intensity and soil moisture trend data.</canvas></div>
      </div>

      <div style='text-align:right;margin-top:-4px;'>
        <button class='btn outline sm' onclick='loadDashboard(true)'>↻ Refresh dashboard</button>
      </div>
    </section>

  </main>
</div>

<script>
// ── State ────────────────────────────────────
const loaded = { 0: false, 1: false, 2: false, 3: false };
let chartTH = null, chartLS = null;
let healthBase = 0;            // the real health score from the latest sensor reading
let careBonusApplied = false;  // did we already add today's +5 "care bonus"? (resets on reload)

// ── Tab switching ────────────────────────────
function switchTab(idx) {
  document.querySelectorAll('.tab-panel').forEach((p,i) => p.classList.toggle('active', i===idx));
  document.querySelectorAll('.tab-btn').forEach((b,i) => b.classList.toggle('active', i===idx));
  if (!loaded[idx]) {
    if (idx === 1) loadSensors();
    if (idx === 3) loadDashboard(false);
    loaded[idx] = true;
  }
}

// ── Toast ────────────────────────────────────
function toast(msg) {
  const el = document.getElementById('toast');
  el.textContent = msg;
  el.classList.add('show');
  setTimeout(() => el.classList.remove('show'), 2800);
}

// ── Invoke helper ────────────────────────────
function invoke(fn, payload) {
  return new Promise((res, rej) => {
    google.colab.kernel.invokeFunction(fn, [JSON.stringify(payload)], {})
      .then(r => {
        const d = (r && r.data) ? r.data : {};
        // The callback RETURNs IPython JSON → arrives as application/json (already an object)
        const aj = d['application/json'];
        if (aj != null) {
          return res(typeof aj === 'string' ? JSON.parse(aj) : aj);
        }
        // Fallback: text/plain may hold a JSON string
        const txt = d['text/plain'];
        if (txt && typeof txt === 'string') {
          try { return res(JSON.parse(txt)); } catch {}
        }
        res({});
      }).catch(rej);
  });
}

// ══════════════════════════════════════
//  TAB 0 — Upload
// ══════════════════════════════════════
let selectedFile = null;

function handleFile(file) {
  if (!file) return;
  selectedFile = file;
  document.getElementById('fname-input').value = file.name;
  const reader = new FileReader();
  reader.onload = e => {
    const img = document.getElementById('preview-img');
    img.src = e.target.result;
    img.style.display = 'block';
  };
  reader.readAsDataURL(file);
}

function handleDrop(e) {
  e.preventDefault();
  document.getElementById('drop-zone').classList.remove('drag-over');
  const file = e.dataTransfer.files[0];
  if (file && file.type.startsWith('image/')) handleFile(file);
}

async function doUpload() {
  const fname = document.getElementById('fname-input').value.trim() || 'image.jpg';
  const notes = document.getElementById('notes-input').value.trim();
  const msgEl = document.getElementById('upload-msg');
  msgEl.innerHTML = '<span style="color:var(--text-hint);font-size:12px;">Uploading…</span>';
  try {
    const res = await invoke('upload_image', { filename: fname, notes });
    if (res.success) {
      msgEl.innerHTML = '<span class="success-msg">✓ ' + res.filename + ' uploaded successfully.</span>';
      addGalleryItem(fname, notes);
      toast('Image uploaded!');
    }
  } catch (err) {
    msgEl.innerHTML = '<span style="color:#c62828;font-size:12px;">Error: ' + err + '</span>';
  }
}

function addGalleryItem(fname, notes) {
  const g = document.getElementById('gallery');
  if (g.querySelector('p')) g.innerHTML = '';
  const div = document.createElement('div');
  div.className = 'gallery-item';
  div.innerHTML = '<b>' + escHtml(fname) + '</b>' + (notes ? escHtml(notes.substring(0,60)) + (notes.length>60?'…':'') : '<em style="opacity:.6">No notes</em>');
  g.appendChild(div);
}

function clearUpload() {
  document.getElementById('fname-input').value = '';
  document.getElementById('notes-input').value = '';
  document.getElementById('upload-msg').innerHTML = '';
  const img = document.getElementById('preview-img');
  img.src = ''; img.style.display = 'none';
  selectedFile = null;
}

// ══════════════════════════════════════
//  TAB 1 — Sensors
// ══════════════════════════════════════
async function loadSensors() {
  try {
    const data = await invoke('get_sensor_data', {});
    const rows = Array.isArray(data) ? data : [];
    if (rows.length) {
      const last = rows[rows.length - 1];
      document.getElementById('mc-hum').innerHTML   = last.humidity    + '<span class="metric-unit">%</span>';
      document.getElementById('mc-temp').innerHTML  = last.temperature + '<span class="metric-unit">°C</span>';
      document.getElementById('mc-light').innerHTML = last.light       + '<span class="metric-unit">lux</span>';
      document.getElementById('mc-soil').innerHTML  = last.soil        + '<span class="metric-unit">%</span>';
    }
    const tbody = document.getElementById('sensor-tbody');
    tbody.innerHTML = rows.slice().reverse().map(r =>
      '<tr><td>' + escHtml(r.timestamp) + '</td><td>' + r.humidity + '</td><td>' +
      r.temperature + '</td><td>' + r.light + '</td><td>' + r.soil + '</td></tr>'
    ).join('') || '<tr><td colspan="5" style="color:var(--text-hint);text-align:center;">No data</td></tr>';
  } catch(e) {
    document.getElementById('sensor-tbody').innerHTML =
      '<tr><td colspan="5" style="color:#c62828;">Error loading data</td></tr>';
  }
}

function simulateReading() {
  document.getElementById('in-hum').value   = (45  + Math.random() * 20).toFixed(1);
  document.getElementById('in-temp').value  = (18  + Math.random() * 8).toFixed(1);
  document.getElementById('in-light').value = Math.round(200 + Math.random() * 600);
  document.getElementById('in-soil').value  = (30  + Math.random() * 25).toFixed(1);
}

async function saveReading() {
  const payload = {
    humidity:    parseFloat(document.getElementById('in-hum').value),
    temperature: parseFloat(document.getElementById('in-temp').value),
    light:       parseFloat(document.getElementById('in-light').value),
    soil:        parseFloat(document.getElementById('in-soil').value),
  };
  const msgEl = document.getElementById('save-msg');
  try {
    const res = await invoke('add_sensor_reading', payload);
    if (res.success) {
      msgEl.innerHTML = '<span class="success-msg">✓ Reading saved.</span>';
      toast('Sensor reading saved!');
      loadSensors();
    }
  } catch(e) {
    msgEl.innerHTML = '<span style="color:#c62828;font-size:12px;">Error: ' + e + '</span>';
  }
}

// ══════════════════════════════════════
//  TAB 2 — Search
// ══════════════════════════════════════
function quickSearch(term) {
  document.getElementById('search-input').value = term;
  doSearch();
}

async function doSearch() {
  const q = document.getElementById('search-input').value.trim();
  if (!q) { toast('Enter a search query first.'); return; }

  const resEl = document.getElementById('search-results');
  resEl.innerHTML = `
    <div style="display:flex;align-items:center;gap:10px;padding:12px 0;color:var(--text-hint);font-size:13px;">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"
           style="animation:spin 1s linear infinite;flex-shrink:0;">
        <circle cx="12" cy="12" r="10" stroke-opacity=".25"/>
        <path d="M12 2a10 10 0 0 1 10 10" stroke="var(--green-md)"/>
      </svg>
      Searching…
    </div>`;

  try {
    const data = await invoke('search_index', { query: q });
    const ragResults = data.rag_results || [];

    if (!ragResults.length) {
      resEl.innerHTML = `<p style="color:var(--text-hint);font-size:13px;padding:8px 0;">No results found for <em>${escHtml(q)}</em>.</p>`;
      return;
    }

    // Build query words set for bolding
    const qWords = q.toLowerCase().trim().split(' ').filter(w => w.length > 2);

    function boldQuery(text) {
      let out = escHtml(text);
      qWords.forEach(w => {
        try {
          out = out.replace(new RegExp('(' + w + ')', 'gi'),
            '<strong style="color:#1a2e22;font-weight:700;">$1</strong>');
        } catch(e) {}
      });
      return out;
    }

    function truncate(text, max) {
      if (text.length <= max) return text;
      let cut = text.slice(0, max);
      const sp = cut.lastIndexOf(' ');
      if (sp > 0) cut = cut.slice(0, sp);
      return cut + ' …';
    }

    const header = `<p style="font-size:13px;color:#70757a;margin-bottom:16px;">
      About <strong>${ragResults.length}</strong> result${ragResults.length===1?'':'s'} for <strong>${escHtml(q)}</strong>
    </p>`;

    const cards = ragResults.map((r, i) => {
      // fname now holds the real article title (stored in the Firebase index),
      // so use it directly; fall back to a generic label only if it's empty.
      const raw   = r.fname || '';
      const title = raw.trim() || ('Research Paper ' + (i + 1));

      // Extract snippet lines
      const snip = (r.snippet || '').trim();
      const desc = snip ? truncate(snip, 220) : 'No preview available.';

      // Breadcrumb path from filename
      const breadcrumb = raw ? `drive.google.com › ${escHtml(raw)}` : 'drive.google.com › file';

      const chips = (r.terms || []).map(t =>
        `<span class="search-result-chip">${escHtml(t)}</span>`
      ).join('');

      return `
        <div class="search-result-item">
          <div class="search-result-breadcrumb">
            <svg class="search-result-pdf-icon" viewBox="0 0 24 24">
                <path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8zM14 2v6h6M8 13h8M8 17h8M8 9h2v2H8z" fill="currentColor"/>
            </svg>
            <span>${breadcrumb}</span>
          </div>
          <a href="${escHtml(r.link)}" target="_blank"
             class="search-result-link">
            ${escHtml(title)}
          </a>
          <p class="search-result-description">
            ${boldQuery(desc)}
          </p>
          <div class="search-result-chip-container">${chips}</div>
        </div>`;
    }).join('');

    resEl.innerHTML = header + cards;
  } catch(e) {
    resEl.innerHTML = `<p style="color:#c62828;font-size:12px;">Search error: ${e}</p>`;
  }
}
// ══════════════════════════════════════
//  TAB 3 — Dashboard
// ══════════════════════════════════════
async function loadDashboard(force) {
  try {
    const data = await invoke('get_dashboard_data', {});
    const readings = data.readings || [];
    const health   = data.health_score ?? 0;
    const alerts   = data.alerts || [];
    const latest   = data.latest || {};

    // Health score. We remember the "real" score from the sensors as the base,
    // and clear any care bonus from a previous visit (a fresh load = fresh truth).
    healthBase = health;
    careBonusApplied = false;
    setHealthScore(health);

    // Alerts
    document.getElementById('dash-alerts').innerHTML =
      alerts.map(a => '<div class="alert-item">' + escHtml(a) + '</div>').join('') || '<p style="font-size:12px;color:var(--text-hint);">No alerts.</p>';

    // Today's care tasks (sensor-driven checklist) + streak count
    renderTasks(data.tasks || [], data.streak ?? 0);

    // Latest metric cards
    document.getElementById('d-hum').innerHTML   = (latest.humidity    ?? '—') + '<span class="metric-unit">%</span>';
    document.getElementById('d-temp').innerHTML  = (latest.temperature ?? '—') + '<span class="metric-unit">°C</span>';
    document.getElementById('d-light').innerHTML = (latest.light       ?? '—') + '<span class="metric-unit">lux</span>';
    document.getElementById('d-soil').innerHTML  = (latest.soil        ?? '—') + '<span class="metric-unit">%</span>';

    // Charts
    const labels = readings.map(r => r.timestamp);
    const temps  = readings.map(r => r.temperature);
    const hums   = readings.map(r => r.humidity);
    const lights = readings.map(r => r.light);
    const soils  = readings.map(r => r.soil);

    // Chart 1 — Temp + Humidity
    const ctx1 = document.getElementById('chartTH').getContext('2d');
    if (chartTH) { chartTH.destroy(); chartTH = null; }
    chartTH = new Chart(ctx1, {
      type: 'line',
      data: {
        labels,
        datasets: [
          {
            label: 'Temperature °C',
            data: temps,
            borderColor: '#2d6a4f',
            backgroundColor: 'rgba(45,106,79,0.08)',
            tension: 0.35, fill: true,
            pointRadius: 3, pointBackgroundColor: '#2d6a4f',
          },
          {
            label: 'Humidity %',
            data: hums,
            borderColor: '#52b788',
            backgroundColor: 'transparent',
            borderDash: [4,3],
            tension: 0.35, fill: false,
            pointRadius: 3, pointBackgroundColor: '#52b788',
          },
        ]
      },
      options: {
        responsive: true, maintainAspectRatio: false,
        plugins: { legend: { display: false } },
        scales: {
          x: { ticks: { autoSkip: false, maxRotation: 45, font: { size: 10 }, color: '#7a9e8a' }, grid: { color: 'rgba(45,106,79,0.06)' } },
          y: { ticks: { font: { size: 10 }, color: '#7a9e8a' }, grid: { color: 'rgba(45,106,79,0.08)' } },
        }
      }
    });

    // Chart 2 — Light + Soil
    const ctx2 = document.getElementById('chartLS').getContext('2d');
    if (chartLS) { chartLS.destroy(); chartLS = null; }
    chartLS = new Chart(ctx2, {
      type: 'line',
      data: {
        labels,
        datasets: [
          {
            label: 'Light lux',
            data: lights,
            borderColor: '#b7e4c7',
            backgroundColor: 'rgba(183,228,199,0.18)',
            tension: 0.35, fill: true,
            pointRadius: 3, pointBackgroundColor: '#b7e4c7',
          },
          {
            label: 'Soil moisture %',
            data: soils,
            borderColor: '#40916c',
            backgroundColor: 'transparent',
            borderDash: [4,3],
            tension: 0.35, fill: false,
            pointRadius: 3, pointBackgroundColor: '#40916c',
          },
        ]
      },
      options: {
        responsive: true, maintainAspectRatio: false,
        plugins: { legend: { display: false } },
        scales: {
          x: { ticks: { autoSkip: false, maxRotation: 45, font: { size: 10 }, color: '#7a9e8a' }, grid: { color: 'rgba(45,106,79,0.06)' } },
          y: { ticks: { font: { size: 10 }, color: '#7a9e8a' }, grid: { color: 'rgba(45,106,79,0.08)' } },
        }
      }
    });

  } catch(e) {
    console.error('Dashboard error:', e);
  }
}

// Paint the health score number, bar width, colour and label for a given score.
// Pulled out into its own function so both the dashboard load and the care
// bonus can reuse it. `note` is an optional extra hint shown after the label.
function setHealthScore(score, note) {
  const scoreEl = document.getElementById('dash-health');
  const barEl   = document.getElementById('health-bar');
  const lblEl   = document.getElementById('health-label');
  scoreEl.textContent = score;
  barEl.style.width = score + '%';
  let label;
  if (score >= 75) {
    barEl.style.background = '#40916c';
    scoreEl.style.color    = '#2d6a4f';
    label = 'Good — plant is thriving';
  } else if (score >= 50) {
    barEl.style.background = '#f9a825';
    scoreEl.style.color    = '#8a6000';
    label = 'Fair — monitor conditions';
  } else {
    barEl.style.background = '#c62828';
    scoreEl.style.color    = '#c62828';
    label = 'Poor — action needed';
  }
  lblEl.textContent = note ? (label + ' · ' + note) : label;
}

// ── Daily care tasks (dashboard card) ────────
// Draw the task list as a checklist and update the streak number. If the only
// task is the "All good" message we show it as a single happy line instead of
// a checkbox row.
function renderTasks(tasks, streak) {
  document.getElementById('streak-count').textContent = streak;

  const ul = document.getElementById('task-list');
  const allGood = tasks.length === 1 && /^all good/i.test(tasks[0]);
  const noData  = tasks.length === 1 && /^no sensor data/i.test(tasks[0]);

  if (allGood || noData) {
    ul.innerHTML = '<li class="task-all-good">' + (allGood ? '✅ ' : '⚠️ ') + escHtml(tasks[0]) + '</li>';
    document.getElementById('complete-btn').style.display = allGood ? '' : 'none';
    return;
  }

  document.getElementById('complete-btn').style.display = '';
  ul.innerHTML = tasks.map((t, i) =>
    '<li class="task-item" id="task-' + i + '">' +
      '<input type="checkbox" onchange="toggleTask(' + i + ')">' +
      '<span>' + escHtml(t) + '</span>' +
    '</li>'
  ).join('');
}

// strike through a single task when its checkbox is ticked
function toggleTask(i) {
  const li = document.getElementById('task-' + i);
  li.classList.toggle('done', li.querySelector('input').checked);
}

// tell the backend the manager finished today's tasks -> bumps the streak
async function markTasksDone() {
  try {
    const res = await invoke('complete_daily_tasks', {});
    if (res.success) {
      document.getElementById('streak-count').textContent = res.streak;
      document.querySelectorAll('#task-list .task-item').forEach(li => {
        const cb = li.querySelector('input');
        if (cb) cb.checked = true;
        li.classList.add('done');
      });

      // +5 "care bonus" on the displayed health score (capped at 100), once per
      // visit. This is cosmetic — it rewards finishing the tasks and disappears
      // on the next dashboard refresh, when the score goes back to the real
      // sensor-based value.
      if (!careBonusApplied) {
        careBonusApplied = true;
        const boosted = Math.min(100, healthBase + 5);
        setHealthScore(boosted, '+5 care bonus');
      }

      toast('Nice! Streak: ' + res.streak + ' day' + (res.streak === 1 ? '' : 's'));
    }
  } catch (e) {
    toast('Could not save: ' + e);
  }
}

// ── Utility ──────────────────────────────────
function

escHtml(str) {
  return String(str).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;').replace(/"/g,'&quot;');
}
''')